# Build molecule map dataset (Colab)

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough).

Pipeline: HuggingFace SMILES → MACCS/Morgan embeddings → ranking → GPU UMAP → CSV + SQL loader for PostgreSQL.

Start with a small `LIMIT` (e.g. 10_000) to test, then scale up.

In [ ]:
# Install dependencies. If Colab asks to restart, do Runtime → Restart session, then skip this cell.
!pip install -q datasets rdkit-pypi scikit-learn umap-learn tqdm cuml-cu12

## Get the project code

Your pipeline code lives in `backend/` (`app/services/` + `scripts/`). If it is **not pushed to GitHub yet**, use **Option B**.

- **Option A:** `git clone` from GitHub (only works after `app/services/` is committed and pushed).
- **Option B (recommended for now):** On your laptop, run `zip -r backend.zip backend/` from the repo root, upload the zip below, unzip.

In [ ]:
import os
from pathlib import Path

USE_UPLOAD = True  # False = clone from GitHub

if USE_UPLOAD:
    from google.colab import files
    if not Path("backend/app/services").exists():
        print("Upload backend.zip (zip the backend/ folder from your repo)")
        uploaded = files.upload()
        !unzip -o backend.zip
    os.chdir("backend")
else:
    REPO_URL = "https://github.com/schwedeErik/MoleculeViewer.git"
    if not Path("MoleculeViewer").exists():
        !git clone {REPO_URL}
    os.chdir("MoleculeViewer/backend")

print("Working directory:", os.getcwd())
assert Path("app/services/smiles_processing.py").exists(), "Missing app/services — check zip or clone"

## Configuration

In [ ]:
REPO_ID = "edmanft/zinc250k"   # HuggingFace dataset
COLUMN = "smiles"
LIMIT = 10_000                  # start small; try 250_000 or 1_000_000 when ready
METHOD = "maccs"                # maccs | morgan
METRIC = "euclidean"            # euclidean or cosine (for dense embeddings)
OUT_DIR = "./out"
NAME = "molecules"

## Run pipeline (embed → rank → UMAP → export)

In [ ]:
from pathlib import Path

from app.services.smiles_downloader import download_smiles_database_huggingface
from app.services.smiles_processing import EmbeddingMethod, process_and_export

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"{NAME}.csv"
sql_path = out_dir / f"{NAME}_load.sql"

print(f"Downloading up to {LIMIT:,} SMILES from {REPO_ID} …")
smiles = download_smiles_database_huggingface(REPO_ID, column_name=COLUMN, limit=LIMIT)
print(f"  got {len(smiles):,} SMILES")

print(f"Embedding ({METHOD}), ranking, UMAP ({METRIC}) …")
n = process_and_export(
    smiles,
    EmbeddingMethod(METHOD),
    csv_path=str(csv_path),
    sql_path=str(sql_path),
    umap_kwargs={"metric": METRIC, "n_neighbors": 15, "min_dist": 0.1},
)

print(f"Done — wrote {n:,} rows")
print(f"  CSV: {csv_path}")
print(f"  SQL: {sql_path}")

## Download results

On your machine (with Postgres running):

```bash
psql "$DATABASE_URL" -f molecules_load.sql
```

In [ ]:
from google.colab import files

files.download(str(csv_path))
files.download(str(sql_path))